<a href="https://colab.research.google.com/github/SawadaAkimichi37/lecture-ai-engineering-2026/blob/lec02/lora_format_tuning_%E7%99%BA%E5%B1%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第2回 演習: LoRA で回答形式を学習し、学習前後を比較する

架空の講座 FAQ を題材に、小さな指示チューニング済みモデルを LoRA で短時間学習し、学習の前後で **何が変わり、何が変わらないか** を比べます。評価は **形式遵守率**（`answer` と `needs_review` の JSON を守れたか）と **内容正解率** に分けます。

**この演習で確かめること**

1. プロンプトで形式を指示しても、小さなモデルは守れないことがある（セクション 3、学習前）。
2. LoRA で数分学習すると、形式はほぼ守れるようになる（セクション 5、学習後）。
3. 形式が整っても、FAQ に無いことを聞かれたときの判断（確認を求めるか、断定してしまうか）までは保証されない（セクション 5、6）。
4. 発展: 同じモデルを 4bit 量子化で読み込むと、学習しなくても出力が変わる（セクション 8）。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、データ生成、モデル読み込み、学習前評価 | 約 5 分 |
| 4〜7 | 学習、学習後評価、知識質問、比較表と提出 | 約 10 分（学習 4〜5 分を含む） |
| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |

授業前に 0〜3 を実行して環境が動くことを確かめておくと、当日の詰まりが減ります（任意）。Colab はセッションが切れると変数が消えるので、その場合は当日あらためて 0 から実行します。

**使うもの**: Google Colab（T4 GPU、無料枠）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。データはこのノートブックが生成する架空データで、実在の締切や URL は含みません。

**読み方**: 各セクションの説明に「やること」「見るところ」「うまくいかないとき」を書いています。セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。

版: 1.1（2026-10-18）

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = True # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 架空 FAQ と教師データの生成

**やること**: 5 項目の架空 FAQ から、学習用（train）、検証用（valid）、評価用（test）のデータを作ります。

モデルへの入力は「FAQ ＋ 質問」、望ましい出力は `{"answer": <文字列>, "needs_review": <真偽値>}` の JSON 1 行です。

- FAQ に根拠がある質問には、FAQ の文で答え、`needs_review: false` にする。
- FAQ に無い質問には、「この FAQ には記載がありません。運営に確認が必要です。」と答え、`needs_review: true` にする。

**データの分け方**

- 同じ FAQ 項目を聞く質問の言い換えを、**テンプレート単位**で train / valid / test に振り分けます。学習で見た文がそのまま test に出ないようにするためです。
- test は 21 問（根拠あり 15、根拠なし 6）です。件数を確保するため、test にだけ使う質問を別に用意しています。
- 正解（`needs_review` の値と、答えに含まれるべき語句）は `gold` に分けて持ち、モデルには渡しません。
- train は 14 件（根拠あり 10、根拠なし 4）、valid は 7 件です。FAQ が 5 項目しかないので学習データは少なく、同じ例を何回も学習することになります。何回学習するかは、データを複製して増やすのではなく、セクション 4 のエポック数（`num_train_epochs`）で決めます。データのファイルには実例だけを入れ、学習量は学習の設定で持つのが基本です。

**見るところ**: `train=14 valid=7 test=21 gold=21` と、学習例の 1 件（質問と、望ましい出力）が表示されます。質問が train / valid / test のどれかで重複していると、ここでエラーになります。

In [ ]:
FAQ = {
    "deadline": "課題の提出期限は 9 月 30 日 23:59（日本時間）です。",
    "destination": "提出先は学習ポータルの「課題」欄です。",
    "format": "提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。",
    "office_hours": "質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。",
    "late": "期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# 根拠あり: (FAQ キー, 質問テンプレート群, 正解の answer に含まれるべき語)
GROUNDED = {
    "deadline": (["課題の提出期限はいつですか", "いつまでに出せばいいですか", "締切を教えてください", "提出の期限は？"], ["9 月 30 日", "23:59"]),
    "destination": (["提出先はどこですか", "どこにアップロードしますか", "課題はどこから出しますか", "提出場所は？"], ["学習ポータル", "課題"]),
    "format": (["提出形式を教えてください", "ファイル名の決まりはありますか", "何のファイル形式で出しますか", "PDF 以外でも出せますか"], ["PDF"]),
    "office_hours": (["質問はいつできますか", "オフィスアワーはいつですか", "先生に質問する時間はありますか", "相談できる時間帯は？"], ["火曜", "17:00"]),
    "late": (["遅れて出したらどうなりますか", "期限を過ぎた場合は受け付けますか", "遅延提出のルールは？", "締切後の扱いを教えてください"], ["48 時間", "減点"]),
}
# 根拠なし: FAQ に無いので needs_review=true が正解
UNGROUNDED = [
    "再提出はできますか", "欠席した回の扱いを教えてください", "他の受講者の点数は見られますか",
    "成績はいつ出ますか", "提出期限は延長できますか", "グループでの提出は可能ですか",
    "課題の配点はどうなっていますか", "講義の録画は公開されますか",
]

# test 専用の追加質問（train / valid には入れない）
TEST_EXTRA_GROUNDED = {
    "deadline": ["提出期限の日付と時刻を教えてください", "課題の締切は何月何日の何時ですか"],
    "destination": ["課題の提出はどこで行いますか", "提出物を出す先を教えてください"],
    "format": ["提出するファイルの形式は何ですか", "提出ファイルは何形式で、いくつ出しますか"],
    "office_hours": ["オフィスアワーの曜日と時間を教えてください", "質問を受け付けてもらえるのは何曜日の何時からですか"],
    "late": ["提出が締切に間に合わなかったらどうなりますか", "期限後に提出した課題は受理されますか"],
}
TEST_EXTRA_UNGROUNDED = [
    "課題の文字数に制限はありますか", "提出した課題へのフィードバックはもらえますか",
    "参考文献の書き方に指定はありますか", "オフィスアワーは事前予約が必要ですか",
]
UNGROUNDED_ANSWER = "この FAQ には記載がありません。運営に確認が必要です。"

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # テンプレートの先頭 2 つを train、3 つ目を valid、4 つ目を test に固定（言い換えを跨がせない）
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # 根拠なしの質問は 4 問ごとに train 2 / valid 1 / test 1 に振る
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["確認"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["確認"]}

# train は 14 件（根拠あり 10、根拠なし 4）。同じ例を何回学習するかは、データを複製せず、セクション 4 の num_train_epochs で決める
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "質問が train / valid / test のどこかで重複しています。テンプレートを見直してください"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("学習例の 1 件 | 質問:", train[0]["question"], "| 望ましい出力:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 21 問に答えさせて採点します。

プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「9 月 30 日」と「23:59」）が `answer` に含まれること。

内容の判定には `needs_review` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `needs_review` が欠けているはずです。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。
  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。
  - 自分のモデルやデータを Hub にアップロードするとき。
  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。
  
  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。

**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_lab/eval_before.json` にも保存されています（左のファイル一覧から開けます）。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = "あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の 21 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

## 4. LoRA 学習（T4 で 4〜5 分）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。

### 学習の実行

**やること**: train の 14 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（40 ステップ）学習します。学習した adapter は `adapter/` に保存します。

- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 14 件なので 1 エポックは 2 ステップ（8 件と 6 件）で、20 エポックで 40 ステップです。各例をちょうど 20 回学習します。学習率の warmup と減衰（cosine）はこの 40 ステップに対して効きます。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。
- `Training Loss` の表（10 ステップごと）: 1.5 前後から始まり、40 ステップで 0.01〜0.02 まで下がります。下がらないまま終わった場合は、設定を変えていないか確かめてください。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`: T4 で 240〜290 秒です。ステップ数とエポックは、学習データ 14 件 × 20 周が上の計算どおりに実行されたことの確認です。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから 40 ステップ（20 エポック）まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師が同じノートブックを T4 で実行した結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 21 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - 根拠なしの質問（正解の `needs_review` が `True`）に、FAQ の別項目を当てはめて断定した。
  - 根拠ありの質問（正解が `False`）に、「記載がありません」と確認要求で逃げた。

学習後の回答は、教師データの文（FAQ の原文）をほぼそのまま返す形になります。形式と文面は揃いますが、「FAQ に根拠があるか」の判断まで学習できたとは限りません。

seed を固定しても、GPU 上の学習は完全には再現しません。形式遵守率と内容正解率の割合はほぼ同じでも、どの質問が残るかは実行ごとに変わることがあります。隣の人と「残った質問」が違っても異常ではないので、失敗の型で比べてください。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. 知識を問う質問（形式を学んでも知識は保証されない）

**やること**: 形式の実験とは別に、FAQ にも学習データにも無い事実を 3 問尋ねます（対照として FAQ にある質問を 1 問加えます）。望ましい応答は「確認が必要（`needs_review: true`）」です。

応答を 3 つに分類します。

| 分類 | 意味 |
| --- | --- |
| `format_error` | JSON の形式を守れていない（学習前に多い） |
| `needs_review` | 「記載がありません。確認が必要です」と返した（望ましい応答） |
| `asserted` | `needs_review: false` で言い切った。FAQ に無い質問に対してなら、根拠のない断定（創作）です |

**見るところ**: 学習前は形式違反になりやすく、学習後は「確認が必要」と「断定」の両方が出ることがあります。出力の原文も表に出すので、断定した場合に何を根拠にしたように見えるか（FAQ の別項目を流用しているか）を読んでください。「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。

学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。

In [ ]:
KNOWLEDGE_QUESTIONS = [
    "この講座の最終課題の締切はいつですか",       # FAQ に無い（架空）
    "担当教員の名前は何ですか",                     # FAQ に無い
    "次回の講義のテーマは何ですか",                 # FAQ に無い
    "課題の提出期限はいつですか",                   # FAQ にある（対照）
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": q, "FAQ にある": q in gold, "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                 "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード。`USE_DRIVE = True` なら Drive にもコピーされます）
- セクション 5 の比較表と、残った失敗の説明（どの型の失敗か）
- セクション 6 の結果（確認要求に変わった応答、断定が残った応答）
- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

**やること**: 同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。QLoRA は、この 4bit の基盤モデルに LoRA を付けて学習する方法です。ここでは学習はしません。

- bitsandbytes は CUDA の GPU が前提なので、GPU ランタイムでだけ動きます。導入に 1 分ほどかかります。
- 4bit の読み込みは `device_map="auto"` で GPU に配置します（bitsandbytes の読み込み方式に合わせています）。
- 4bit 側で学習まで試す場合は、`prepare_model_for_kbit_training(model_4bit)` を通してからセクション 4 と同じ手順で学習し、学習前後とも 4bit の基盤モデルで評価します。

**見るところ**

- 重みのメモリ: 32bit の約 3GB に対して 4bit は 1GB 未満になります。
- 比較表と質問ごとの表: 学習していないので、32bit の学習前（`before`）との差は量子化の影響です。良くなる方向に変わる質問もあれば、`needs_review` が `"false"`（文字列）になって型検査で落ちる、根拠なしの質問に「はい」「なし」と断定する、といった出力も見られます。方向が一定でないことを確かめてください。

In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                         bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "4bit・学習前の出力": r["raw"],
                       "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式の失敗理由": r["format_reason"]} for r in q_rows]))

## 9. 発展: 用語集で学習し、知識を覚えられるかを確かめる

本編では「形式は学習できるが、知識は保証されない」ことを見た。ここでは知識（用語の定義）そのものを学習させて、次の 3 つを確かめる。

1. 覚えたか: 基盤モデルが知らない講座固有の事実を、学習後に答えられるか
2. 応用が利くか: 学習データと違う聞き方でも答えられるか
3. 副作用はないか: 用語集に無い語に、講座の話を作って答えないか（学習データは正例のみ）

予想（実行前に書く）
- 1:与えた内容は覚えるが、元答えれた内容をこたえられなくなりそう
- 2:答えることができる例が増える
- 3:わかったふりをして回答をする

評価で意識した 3 つの観点（test の作り方はこれに合わせる）

| 観点 | 何を見るか | 使う質問（9-2） | 学習前→学習後で悪い兆候 |
|---|---|---|---|
| A. 忘却 | 実在する用語で、もともと答えられたはずの精度が下がっていないか | 用語集に無い実在の用語（RAG など）、一般の用語 | 正しかった説明が崩れる・講座の話にすり替わる |
| B. 教えた内容 | 教えた内容（特に講座固有の事実）を答えられるか。聞き方を変えても答えられるか | 講座固有の用語（学習データと違う聞き方で質問） | 学習後も must_include が出てこない |
| C. 分かったふり | もともと知らないことに、分かったふりをして答えることが増えるか | 架空の講座用語（lecNN-end タグ など） | 存在しない用語に定義を作る・講座の話を作る |

データについて気づいたこと
- 学習データは 58 語 × 2 通りの聞き方（「〜」とは何ですか？ / 〜について簡単に説明してください。）で、2 通りの答えは 58 語すべて同じ文。「簡単に」が答えに反映されていない
- 正例のみで、「用語集に無い」と答える例が無い（観点 C が起きやすい条件）

In [1]:
# 9-0. 準備（セクション 9 だけで完結させるため、上のセクション 0・1・3 から必要な部分をまとめたもの）
#   実行のしかた: このセルを選んで「ランタイム」→「選択したセル以降を実行」。FAQ のセクション（2〜8）は実行しなくてよい
#   FAQ の学習（セクション 4）を同じセッションで実行した後なら、先に「ランタイム」→「セッションを再起動」する（model に adapter が付いたままになるため）

# --- セクション 0: ライブラリの導入（動作確認済みの版に固定） ---
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}
import transformers, peft, trl, accelerate, datasets, torch

# --- セクション 1: 設定と Drive のマウント ---
from pathlib import Path
import os, json, time
CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42, "warmup_steps": 6, "lr": 2e-4, "lora_r": 16, "lora_alpha": 32, "max_seq_len": 512,
    "gen": {"do_sample": False},
    "work_dir": "/content/qlora_lab",
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)

# --- セクション 3 の 1 つ目: モデルの読み込み（量子化なし・32bit） ---
from transformers import AutoTokenizer, AutoModelForCausalLM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
if "model" not in globals() or hasattr(model, "peft_config"):  # 読み込み済みで adapter も無ければ読み直さない
    model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print({m.__name__: m.__version__ for m in [transformers, peft, trl, torch]}, "| device:", DEVICE)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
Mounted at /content/drive


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'torch': '2.11.0+cu130'} | device: cuda


In [2]:
# 9-1. 用語集データの読み込みと中身の確認
from collections import Counter
import re, shutil
import pandas as pd
pd.set_option("display.max_colwidth", None)  # 出力の原文を省略せずに表示する
GLOSS_PATH = Path("/content/drive/MyDrive/Colab Notebooks/aie2026/glossary_sft.jsonl")
assert GLOSS_PATH.exists(),f"見つかりません:{GLOSS_PATH}"
gloss = [json.loads(l) for l in GLOSS_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]

GW = WORK / "glossary"; GW.mkdir(exist_ok=True)           # 発展の結果はここにまとめる
GDRIVE = Path(CONFIG["drive_dir"]) / "glossary"

# 用語 → 模範の定義文（「～」とは何ですか？の行から穴の中身を取り出す
GLOSSARY = {}
for r in gloss:
    m = re.fullmatch(r"「(.+)」とは何ですか？",r["messages"][0]["content"])
    if m: GLOSSARY[m.group(1)] = r["messages"][-1]["content"]
lens = [len(a) for a in GLOSSARY.values()]

print(f"件数: {len(gloss)} | 用語: {len(GLOSSARY)} 語 | 定義文の長さ: {min(lens)}〜{max(lens)} 字")
print("役割の並び:", Counter(tuple(m["role"] for m in r["messages"]) for r in gloss))
for r in gloss[:2]: print(r["messages"])

件数: 116 | 用語: 58 語 | 定義文の長さ: 45〜212 字
役割の並び: Counter({('user', 'assistant'): 116})
[{'role': 'user', 'content': '「AIエンジニアリング」とは何ですか？'}, {'role': 'assistant', 'content': '学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。'}]
[{'role': 'user', 'content': 'AIエンジニアリングについて簡単に説明してください。'}, {'role': 'assistant', 'content': '学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。'}]


In [3]:
# 9-2. test と gold の作成（観点 A・B・C を確かめられる量にする。★ は自分で見直して決める）
#   must_include は「要点のリスト」。各要点は「書き方の候補のリスト」で、候補のどれか 1 つが入っていればその要点は ○
#   例: [["4ビット", "4bit"], ["量子化"]] → 「4ビット か 4bit」と「量子化」の両方が入っていれば内容 ○（言い回しの違いで × にしないため）

# --- 観点 B: 教えた内容を答えられるか（用語集にある用語。2 通りの聞き方の両方で聞く → 22 語 × 2 = 44 問） ---
GROUNDED_TEST = {
    # 一般: 基盤モデルも事前学習で知っていそう（対照）
    "トークン": ("一般", [["最小単位", "最小の単位"]]),
    "コンテキストウィンドウ": ("一般", [["上限", "最大"]]),
    "ハルシネーション": ("一般", [["もっともらしく", "事実と異なる", "存在しない"]]),
    "LoRA": ("一般", [["低ランク", "low-rank"]]),
    "QLoRA": ("一般", [["4ビット", "4bit", "4 ビット", "4-bit"], ["量子化"]]),
    "エージェントループ": ("一般", [["ツール"], ["繰り返"]]),
    "Codespaces secret": ("一般", [["環境変数"]]),  # ★ 一般か講座固有か迷う
    # 講座固有: 用語集で学ばないと知りえない（ファイル名・パッケージ名など、言い換えられない語句を要点にする）
    "APIキー": ("講座固有", [["GEMINI_API_KEY"]]),
    "ブランチ": ("講座固有", [["lec01"]]),
    "プルリクエスト（PR）": ("講座固有", [["fork"], ["main"]]),
    "Pi": ("講座固有", [["@earendil-works/pi-coding-agent"]]),
    "Pydantic AI": ("講座固有", [["第3回"]]),
    "宿題の3レベル": ("講座固有", [["基礎"], ["実装"], ["発展"]]),
    "answer.md": ("講座固有", [["lectures/NN/answer.md"]]),
    "既定モデル": ("講座固有", [["gemini-3.5-flash-lite"]]),
    "lecNN-start タグ": ("講座固有", [["演習開始"]]),
    "MCP（Model Context Protocol）": ("講座固有", [["第5回"]]),
    "MCPサーバー": ("講座固有", [["app/mcp_server.py"]]),
    "mcpServers設定": ("講座固有", [["app/mcp_servers.json"]]),
    "ツールの名前空間": ("講座固有", [["glossary_search_glossary"]]),
    "fetchサーバー": ("講座固有", [["mcp-server-fetch"]]),
    "Human-in-the-loop": ("講座固有", [["requires_approval"]]),
}

# --- 観点 A: 忘却（用語集に無い実在の用語。もともと答えられたはずの精度が下がらないか → 10 問） ---
#   正解の語句は自分で決める
REAL_UNSEEN = {
    "RAG（検索拡張生成）": [["検索"], ["生成"]],
    "埋め込み（Embedding）": [["ベクトル"]],
    "Transformer": [["注意", "アテンション", "Attention"]],
    "アテンション（注意機構）": [["重み", "関連", "注目"]],
    "ベクトルデータベース": [["ベクトル"], ["検索", "類似"]],
    "RLHF": [["人間", "人の"], ["フィードバック", "評価"]],
    "DPO": [["選好", "好み", "preference"]],
    "知識蒸留": [["教師", "大きな"], ["生徒", "小さな"]],
    "プロンプトインジェクション": [["指示", "命令"], ["攻撃", "乗っ取", "悪意"]],
    "Few-shot プロンプト": [["例"]],
}

# --- 観点 C: 分かったふり（架空の講座用語。存在しないので「わからない」と言えれば正しい → 8 問） ---
FICTIONAL = [
    "lecNN-end タグ", "宿題の4レベル", "環境変数 AIE_TEMPERATURE", "app/eval_server.py",
    "glossary_update_glossary ツール", "Codespaces vault", "report.md（AIE2026 の提出ファイル）", "第9回の演習",
]

# 学習データの聞き方（「〜」とは何ですか？ / 〜について簡単に説明してください。）とは違う言い方
TEST_TEMPLATES = ["{t}って何？", "{t}の意味を一言で教えてください。"]

g_test = []
for t, (kind, must) in GROUNDED_TEST.items():
    assert t in GLOSSARY, f"{t} が用語集に無い（用語名の表記を確認）"
    assert all(any(w in GLOSSARY[t] for w in group) for group in must), f"{t} の要点が定義文に無い: {must}"
    for tpl in TEST_TEMPLATES:  # 同じ用語を 2 通りの聞き方で聞く（片方だけ正解なら、聞き方に弱いとわかる）
        g_test.append({"term": t, "question": tpl.format(t=t), "kind": kind, "must_include": must})
for i, (t, must) in enumerate(REAL_UNSEEN.items()):
    assert t not in GLOSSARY, f"{t} は用語集にある"
    g_test.append({"term": t, "question": TEST_TEMPLATES[i % 2].format(t=t), "kind": "実在・用語集外", "must_include": must})
for i, t in enumerate(FICTIONAL):
    assert t not in GLOSSARY, f"{t} は用語集にある"
    g_test.append({"term": t, "question": TEST_TEMPLATES[i % 2].format(t=t), "kind": "架空", "must_include": []})

# ⑥ 対照: 講座固有の語を、学習データと同じ聞き方でも聞く（15 問）
# ここで × なら「覚えていない」、ここで ○ なのに上の 2 通りが × なら「覚えたが応用が利かない」と切り分けられる
for t, (kind, must) in GROUNDED_TEST.items():
    if kind == "講座固有":
        g_test.append({"term": t, "question": f"「{t}」とは何ですか？", "kind": "講座固有・学習時の聞き方", "must_include": must})

# test の質問が学習データに入っていないこと（⑥ の対照はわざと学習データと同じ聞き方なので除く）（入っていると「聞き方を変えても答えられるか」を測れない）
train_questions = {r["messages"][0]["content"] for r in gloss}
assert not any(e["question"] in train_questions for e in g_test if e["kind"] != "講座固有・学習時の聞き方"), "test の質問が学習データと重複"

(GW / "test.json").write_text(json.dumps(g_test, ensure_ascii=False, indent=2), encoding="utf-8")
print("問数:", len(g_test), dict(Counter(e["kind"] for e in g_test)))
display(pd.DataFrame(g_test)[["question", "kind", "must_include"]])


問数: 77 {'一般': 14, '講座固有': 30, '実在・用語集外': 10, '架空': 8, '講座固有・学習時の聞き方': 15}


,question,kind,must_include
0,トークンって何？,一般,"[[最小単位, 最小の単位]]"
1,トークンの意味を一言で教えてください。,一般,"[[最小単位, 最小の単位]]"
2,コンテキストウィンドウって何？,一般,"[[上限, 最大]]"
3,コンテキストウィンドウの意味を一言で教えてください。,一般,"[[上限, 最大]]"
4,ハルシネーションって何？,一般,"[[もっともらしく, 事実と異なる, 存在しない]]"
...,...,...,...
72,「MCPサーバー」とは何ですか？,講座固有・学習時の聞き方,[[app/mcp_server.py]]
73,「mcpServers設定」とは何ですか？,講座固有・学習時の聞き方,[[app/mcp_servers.json]]
74,「ツールの名前空間」とは何ですか？,講座固有・学習時の聞き方,[[glossary_search_glossary]]
75,「fetchサーバー」とは何ですか？,講座固有・学習時の聞き方,[[mcp-server-fetch]]


In [4]:
# 9-3. 採点関数と、その動作確認（モデルはまだ使わない）
G_GEN_TOKENS = 192  # 生成の最大トークン数。定義文は最長 212 字なので、FAQ（96）より長めにとる
# 9 章のシステムプロンプト。学習データ（9-5）と評価（generate_g）の両方に同じものを入れる
# 文体や講座の話は書かない（文体の変化を学習の効果として測るため）。足すのは「知らなければそう言う」指示だけ
G_SYSTEM = "質問された用語の意味を答えてください。知らない用語や、実在するか確かめられない用語には、定義を作らず「その用語はわかりません。」とだけ答えてください。"
G_TAG = "sys" # 結果ファイル名の接尾辞。system なしの前回の結果（eval_glossary_before.json など）を上書きしない
COURSE_MARKERS = ["本講座", "AIE2026", "講座repo", "第2回", "第3回", "第5回"]  # これが出たら「講座の話に結び付けた」
# system で断り方を「その用語はわかりません。」に固定したので、この語だけで判定する（「確認」「不明」は定義文にも出るので外した）
UNSURE_MARKERS = ["わかりません", "分かりません"]           # これが出たら「わからないと言えた」

def generate_g(model, question):
    '''学習データ（9-5）と同じ system を付けて、質問に答えさせる'''
    msgs = [{"role": "system", "content": G_SYSTEM}, {"role": "user", "content": question}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=G_GEN_TOKENS, pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def check_style(text):
    '''用語集の文体か: 常体・改行や Markdown 記号なし・20〜250 字（学習データは 45〜212 字）'''
    reasons = []
    if re.search(r"(です|ます)[。！]", text): reasons.append("敬体")
    if "\n" in text or re.search(r"[#*`]", text): reasons.append("改行・記号")
    if not 20 <= len(text) <= 250: reasons.append(f"長さ{len(text)}")
    return not reasons, ",".join(reasons)

def check_content(text, ex):
    '''観点 A・B: 要点ごとに、書き方の候補のどれか 1 つが入っていれば ○。全部の要点が ○ なら内容 ○（架空は None）'''
    if ex["kind"] == "架空": return None
    return all(any(w in text for w in group) for group in ex["must_include"])

def check_unsure(text, ex):
    '''「わかりません」と答えたか。C（架空）では ○ が正しい。A・B では ○ は断りすぎ'''
    # 断りすぎを見るため、A・B（実在の語・用語集の語）でも判定する
    return any(w in text for w in UNSURE_MARKERS)

def check_course_link(text, ex):
    '''観点 A・C: 用語集に無い語を、講座の話に結び付けたか'''
    if ex["kind"] not in ("実在・用語集外", "架空"): return None
    return any(w in text for w in COURSE_MARKERS)

def score(text, ex):
    ok_style, reason = check_style(text)
    return {"style_ok": ok_style, "style_reason": reason, "content_ok": check_content(text, ex),
            "unsure_ok": check_unsure(text, ex), "course_link": check_course_link(text, ex)}

# 動作確認 1: 模範の定義文は、文体も内容も全部 ○ になるはず（○ にならなければ採点関数の側がおかしい）
grounded = [e for e in g_test if e["kind"] in ("一般", "講座固有")]
s = [score(GLOSSARY[e["term"]], e) for e in grounded]
print("模範解答 | 文体 ○:", sum(x["style_ok"] for x in s), "/", len(s), "| 内容 ○:", sum(x["content_ok"] for x in s), "/", len(s))

# 動作確認 2: 判定が見分けられるか
fake = {"kind": "架空", "must_include": []}
print(check_style("**トークン**とは、テキストを分割した単位です。\n- 例: 単語"))  # → (False, '敬体,改行・記号')
print(score("AIE2026で提出に使うファイル。lectures/NN/ に置く。", fake))      # → unsure_ok False（分かったふり）, course_link True
print(score("その用語はわかりません。", fake))                  # → unsure_ok True（わからないと言えた）


模範解答 | 文体 ○: 44 / 44 | 内容 ○: 44 / 44
(False, '敬体,改行・記号')
{'style_ok': True, 'style_reason': '', 'content_ok': None, 'unsure_ok': False, 'course_link': True}
{'style_ok': False, 'style_reason': '長さ12', 'content_ok': None, 'unsure_ok': True, 'course_link': False}


In [5]:
# 9-4. 学習前評価（必ず 9-5 の学習より前に実行する。LoRA は model に直接差し込まれるため、学習後の model では学習前を測れない）
assert not hasattr(model, "peft_config"), "model に adapter が付いています。再起動して 0 → 1 → 3 の 1 つ目 → 9-1 から実行し直してください"

def evaluate_g(model, label):
    '''g_test の全問を生成して採点し、観点ごとの要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows, t0 = [], time.time()
    for i, ex in enumerate(g_test):
        raw = generate_g(model, ex["question"])
        rows.append({**ex, "raw": raw, **score(raw, ex)})
        if (i + 1) % 10 == 0: print(f"  {i + 1}/{len(g_test)} 問（{time.time() - t0:.0f} 秒）")

    def count(kinds, key):  # 「○の数/問数」で表す（問数が少ないので割合より数で読む）
        rs = [r for r in rows if r["kind"] in kinds]
        return f"{sum(bool(r[key]) for r in rs)}/{len(rs)}"
    summary = {"label": label, "n": len(rows),
               "文体（用語集の語）": count(("一般", "講座固有"), "style_ok"),
               "B 講座固有の内容": count(("講座固有",), "content_ok"),
               "B 講座固有（学習時の聞き方）": count(("講座固有・学習時の聞き方",), "content_ok"),
               "B 一般の内容": count(("一般",), "content_ok"),
               "A 実在・用語集外の内容": count(("実在・用語集外",), "content_ok"),
               "A 講座の話にすり替え": count(("実在・用語集外",), "course_link"),
               "C わからないと言えた": count(("架空",), "unsure_ok"),
               "C 講座の話を作った": count(("架空",), "course_link"),
               "A 断った（断りすぎ）": count(("実在・用語集外",), "unsure_ok"),
               "B 断った（断りすぎ）": count(("一般", "講座固有"), "unsure_ok"),
               "seconds": round(time.time() - t0, 1)}
    path = GW / f"eval_{label}.json"
    path.write_text(json.dumps({"summary": summary, "system": G_SYSTEM, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    if USE_DRIVE:
        GDRIVE.mkdir(parents=True, exist_ok=True); shutil.copy(path, GDRIVE / path.name)
    print(json.dumps(summary, ensure_ascii=False, indent=1))
    return summary, rows

mk = lambda v: "-" if v is None else ("○" if v else "×")
# すでに学習前の結果が保存されていれば、それを読み込む（再起動しても学習前の評価をやり直さなくて済む）
saved_before = next((p for p in [GW / f"eval_glossary_{G_TAG}_before.json", GDRIVE / f"eval_glossary_{G_TAG}_before.json"] if p.exists()), None)
saved = json.loads(saved_before.read_text(encoding="utf-8")) if saved_before else None
if saved and saved.get("system") == G_SYSTEM and [r["question"] for r in saved["rows"]] == [e["question"] for e in g_test]:  # test と system が同じときだけ使う
    g_before_summary, g_before_rows = saved["summary"], saved["rows"]
    print("保存済みの学習前の結果を読み込みました（再生成していません）:", saved_before)
    print(json.dumps(g_before_summary, ensure_ascii=False, indent=1))
else:
    g_before_summary, g_before_rows = evaluate_g(model, f"glossary_{G_TAG}_before")

# 質問ごとの答え。観点 A は、学習前に本当に正しく説明できているかをここで目で確かめる（学習前から誤りなら、学習後に崩れても忘却とは言えない）
display(pd.DataFrame([{"質問": r["question"], "種類": r["kind"], "学習前の出力": r["raw"], "文体": mk(r["style_ok"]),
                       "内容": mk(r["content_ok"]), "わからない": mk(r["unsure_ok"]), "講座化": mk(r["course_link"])}
                      for r in g_before_rows]))


保存済みの学習前の結果を読み込みました（再生成していません）: /content/drive/MyDrive/aie2026/qlora_lab/glossary/eval_glossary_sys_before.json
{
 "label": "glossary_sys_before",
 "n": 77,
 "文体（用語集の語）": "1/44",
 "B 講座固有の内容": "0/30",
 "B 講座固有（学習時の聞き方）": "0/15",
 "B 一般の内容": "0/14",
 "A 実在・用語集外の内容": "4/10",
 "A 講座の話にすり替え": "0/10",
 "C わからないと言えた": "1/8",
 "C 講座の話を作った": "1/8",
 "A 断った（断りすぎ）": "1/10",
 "B 断った（断りすぎ）": "4/44",
 "seconds": 430.0
}


,質問,種類,学習前の出力,文体,内容,わからない,講座化
0,トークンって何？,一般,トークンとは、通信や情報処理において、データを送信する際に使用される単位や要素のことです。\n\n例えば、メール送信では「メール」という言葉が、通信の単位として使われます。また、インターネット通信では「IP」というアドレスが、通信の単位として使われます。\n\nトークンは、通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使,×,×,×,-
1,トークンの意味を一言で教えてください。,一般,トークンは、通信や情報交換において、メッセージを伝達するための単位です。,×,×,×,-
2,コンテキストウィンドウって何？,一般,コンテキストウィンドウは、テキストエディタやコードエディタなどのソフトウェアにおいて、ユーザーが特定の文脈（例：特定のファイル、特定のコード、特定の環境設定など）を参照して編集する機能です。\n\nこの機能は、ユーザーが「この文脈で何を操作するか」を明確にすることで、作業の効率を向上させ、エラーを減らすために設計されています。,×,×,×,-
3,コンテキストウィンドウの意味を一言で教えてください。,一般,コンテキストウィンドウは、ユーザーが特定の文脈や状況に依存して情報を抽出・分析する機能です。,×,×,×,-
4,ハルシネーションって何？,一般,ハルシネーションとは、光の波長が波長を短くする現象です。\n\nこの現象は、光の波長が波長を短くする現象を指します。光の波長が波長を短くする現象は、ハルシネーションと呼ばれます。\n\n光の波長が波長を短くする現象は、ハルシネーションと呼ばれます。,×,×,×,-
...,...,...,...,...,...,...,...
72,「MCPサーバー」とは何ですか？,講座固有・学習時の聞き方,「MCPサーバー」は、現時点では実在する用語ではありません。\n\nMCP（Model Context Protocol）は、大規模言語モデル（LLM）や AI 技術において、モデルとユーザーとの通信を標準化するためのプロトコルとして提案されたものです。しかし、そのプロトコル自体が「サーバー」という概念を持つわけではありません。\n\nMCP 技術は主に以下の 2 つのプラットフォームで実装されています：\n1. **MCP Server**：これは、MCP プロトコルを定義する**サーバー**（サーバー）です。\n2. **MCP Client**：これは、MCP プロトコルを定義するクライアント（クライアント）です。\n\nしたがって、「MCP サーバー」という用語は、MCP プロトコルを定義するサーバーという概念自体を指すのではなく、MCP 技術全体を指す言葉として使われる,×,×,×,-
73,「mcpServers設定」とは何ですか？,講座固有・学習時の聞き方,「mcpServers」は、MCP（Model Context Protocol）のサーバー設定を指します。\n\nこれは、モデルコンテキストプロトコル（MCP）のサーバーを定義するパラメータです。具体的には、モデルコンテキストプロトコルを定義するサーバーのリストを指定するパラメータです。\n\nこの設定は、モデルコンテキストプロトコルを定義するサーバーのリストを指定するパラメータです。,×,×,×,-
74,「ツールの名前空間」とは何ですか？,講座固有・学習時の聞き方,「ツールの名前空間」とは、ソフトウェア開発において、特定のツール（例：Python の `requests`、Java の `Spring Boot` など）の機能や特性を定義する際に使用される概念です。\n\nこの空間には、ツールが持つ特徴や機能、用途などを記述する定義文（定義文）が含まれます。例えば、`requests` ツールは「HTTP 通信を処理する」などの定義文がその名前空間に含まれます。\n\nただし、この用語は実在しているわけではありません。ツールは実在するが、その「名前空間」自体は定義文として存在しないため、この用語はわかりません。,×,×,○,-
75,「fetchサーバー」とは何ですか？,講座固有・学習時の聞き方,「fetchサーバー」は、HTTP 通信において、クライアント（ユーザーやアプリケーション）がサーバーからデータを取得する際に使用される、主に HTTP 1.1 標準の「Fetch」プロトコルを定義するサーバーです。\n\nこのプロトコルは、クライアントがサーバーからデータを取得する際に、サーバーが直接そのデータを返すのではなく、**クライアントがサーバーからデータを取得する際、そのデータを取得する際に使用される**という特徴を持っています。つまり、サーバーが直接データを返すのではなく、クライアントがサーバーからデータを取得する際に使用されるという定義です。\n\nこのプロトコルは、HTTP 1.1 標準の Fetch プロトコルを定義するサーバーです。,×,×,×,-


In [6]:
# 9-5. 用語集で LoRA 学習（正例のみ。116 件 ÷ 1 ステップ 8 件 = 15 ステップ/エポック × 20 エポック = 300 ステップ）
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

G_EPOCHS = 20  # 5 エポック（75 ステップ）では損失が 0.3 前後で下げ止まらず、学習時の聞き方でも講座固有語を 1/15 しか答えられなかった（学習不足）。本編と同じ 20 エポックにする（各例を 20 回学習。T4 で約 16 分）
G_RUN = f"{G_TAG}_e{G_EPOCHS}"  # adapter と学習後の結果のファイル名。5 エポックの結果（eval_glossary_sys_after.json）を上書きせず、9-7 で並べて比べる
train_g = Dataset.from_list([{"messages": [{"role": "system", "content": G_SYSTEM}] + r["messages"]} for r in gloss])  # 評価（generate_g）と同じ system を先頭に付ける。正例のみという条件は変えない
model.config.use_cache = False
# LoRA の設定は本編セクション 4 と同じ（変えたのはデータとエポック数だけ）
lora_g = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft_g = SFTConfig(output_dir=str(GW / f"ckpt_{G_RUN}"), num_train_epochs=G_EPOCHS, per_device_train_batch_size=4, gradient_accumulation_steps=2,
                  learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10,
                  save_steps=50, save_total_limit=2, fp16=False, max_length=CONFIG["max_seq_len"], packing=False,
                  report_to="none", seed=CONFIG["seed"])
trainer_g = SFTTrainer(model=model, args=sft_g, train_dataset=train_g, peft_config=lora_g, processing_class=tok)
trainer_g.model.print_trainable_parameters()
t0 = time.time()
trainer_g.train()
G_TRAIN_SECONDS = round(time.time() - t0)
print(f"学習時間: {G_TRAIN_SECONDS} 秒 | ステップ数: {trainer_g.state.global_step} | エポック: {trainer_g.state.epoch:.1f}")

trainer_g.model.save_pretrained(GW / f"adapter_{G_RUN}")
if USE_DRIVE:
    shutil.copytree(GW / f"adapter_{G_RUN}", GDRIVE / f"adapter_{G_RUN}", dirs_exist_ok=True)
tuned_g = trainer_g.model
tuned_g.config.use_cache = True
_ = tuned_g.eval()  # dropout を切って評価の出力を揺らさない


Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Step,Training Loss
10,2.655305
20,1.476265
30,1.135739
40,0.676652
50,0.385950
60,0.232805
70,0.116660
80,0.097622
90,0.086161
100,0.073852


学習時間: 1043 秒 | ステップ数: 300 | エポック: 20.0


In [7]:
# 9-6. 学習後評価と比較（9-4 と同じ 77 問・同じ生成設定。変えたのは adapter の有無だけ）
g_after_summary, g_after_rows = evaluate_g(tuned_g, f"glossary_{G_RUN}_after")
display(pd.DataFrame([g_before_summary, g_after_summary]).set_index("label").T)  # 観点ごとの「○の数/問数」を学習前後で並べる

pairs = list(zip(g_before_rows, g_after_rows))
# 観点 A（忘却）: 学習前は正しく説明できたのに、学習後に崩れた実在の用語
print("A 忘却（学習前 ○ → 学習後 ×）:", [b["term"] for b, a in pairs if b["kind"] == "実在・用語集外" and b["content_ok"] and not a["content_ok"]])
print("A 学習後に講座の話にすり替えた:", [a["term"] for b, a in pairs if a["kind"] == "実在・用語集外" and a["course_link"]])
# 観点 B（教えた内容）: 学習後も答えられない用語と、聞き方の片方だけ正解の用語
by_term = {}
for a in g_after_rows:
    if a["kind"] in ("一般", "講座固有"): by_term.setdefault(a["term"], []).append(a["content_ok"])
print("B 学習後、2 通りとも × の用語:", [t for t, v in by_term.items() if sum(v) == 0])
print("B 学習後、聞き方の片方だけ ○ の用語:", [t for t, v in by_term.items() if sum(v) == 1])
print("B 退行（学習前 ○ → 学習後 ×）:", [b["question"] for b, a in pairs if b["kind"] in ("一般", "講座固有") and b["content_ok"] and not a["content_ok"]])
# 観点 C（分かったふり）: 学習前は「わからない」と言えたのに、学習後に言えなくなった架空の用語
print("C 学習前は言えた → 学習後は分かったふり:", [b["term"] for b, a in pairs if b["kind"] == "架空" and b["unsure_ok"] and not a["unsure_ok"]])
# ⑥ 対照: 学習時の聞き方で答えられるかで、「覚えていない」と「応用が利かない」を分ける
same = {a["term"]: a["content_ok"] for a in g_after_rows if a["kind"] == "講座固有・学習時の聞き方"}
print("⑥ 学習時の聞き方でも ×（覚えていない）:", [t for t, v in same.items() if not v])
print("⑥ 学習時の聞き方では ○、違う聞き方は 2 通りとも ×（応用が利かない）:", [t for t, v in same.items() if v and sum(by_term.get(t, [])) == 0])
print("⑥ 学習時の聞き方では ○、違う聞き方の片方だけ ○:", [t for t, v in same.items() if v and sum(by_term.get(t, [])) == 1])
# 「一言で」の指示に従うか: 聞き方ごとの答えの平均の長さ（学習データでは「簡単に」が答えに反映されていなかった）
for tpl in TEST_TEMPLATES:
    key = tpl.format(t="")
    lb = [len(r["raw"]) for r in g_before_rows if r["question"].endswith(key)]
    la = [len(r["raw"]) for r in g_after_rows if r["question"].endswith(key)]
    print(f"平均の長さ「{key}」| 学習前 {sum(lb) / len(lb):.0f} 字 → 学習後 {sum(la) / len(la):.0f} 字")

# 質問ごとの答えを学習前後で並べる。機械の判定と自分の判断が食い違うものがないかを目で確かめる
display(pd.DataFrame([{"質問": b["question"], "種類": b["kind"], "学習前の出力": b["raw"],
                       "学習前 内容/わからない/講座化": f"{mk(b['content_ok'])}/{mk(b['unsure_ok'])}/{mk(b['course_link'])}",
                       "学習後の出力": a["raw"],
                       "学習後 内容/わからない/講座化": f"{mk(a['content_ok'])}/{mk(a['unsure_ok'])}/{mk(a['course_link'])}"}
                      for b, a in pairs]))

result_g = {"config": {**{k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir", "num_train_epochs", "max_new_tokens")},
                       "num_train_epochs": G_EPOCHS, "max_new_tokens": G_GEN_TOKENS},
            "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
            "data": {"file": GLOSS_PATH.name, "n_train": len(gloss), "n_terms": len(GLOSSARY), "n_test": len(g_test)},
            "system": G_SYSTEM, "train_seconds": G_TRAIN_SECONDS, "before": g_before_summary, "after": g_after_summary}
(GW / f"result_glossary_{G_RUN}.json").write_text(json.dumps(result_g, ensure_ascii=False, indent=2), encoding="utf-8")
if USE_DRIVE:
    for name in [f"result_glossary_{G_RUN}.json", "test.json"]:
        shutil.copy(GW / name, GDRIVE / name)
    print("Drive に保存しました:", GDRIVE)


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


  10/77 問（41 秒）
  20/77 問（75 秒）
  30/77 問（115 秒）
  40/77 問（165 秒）
  50/77 問（201 秒）
  60/77 問（234 秒）
  70/77 問（270 秒）
{
 "label": "glossary_sys_e20_after",
 "n": 77,
 "文体（用語集の語）": "44/44",
 "B 講座固有の内容": "30/30",
 "B 講座固有（学習時の聞き方）": "15/15",
 "B 一般の内容": "14/14",
 "A 実在・用語集外の内容": "2/10",
 "A 講座の話にすり替え": "3/10",
 "C わからないと言えた": "0/8",
 "C 講座の話を作った": "5/8",
 "A 断った（断りすぎ）": "0/10",
 "B 断った（断りすぎ）": "0/44",
 "seconds": 302.0
}


label,glossary_sys_before,glossary_sys_e20_after
n,77,77
文体（用語集の語）,1/44,44/44
B 講座固有の内容,0/30,30/30
B 講座固有（学習時の聞き方）,0/15,15/15
B 一般の内容,0/14,14/14
A 実在・用語集外の内容,4/10,2/10
A 講座の話にすり替え,0/10,3/10
C わからないと言えた,1/8,0/8
C 講座の話を作った,1/8,5/8
A 断った（断りすぎ）,1/10,0/10


A 忘却（学習前 ○ → 学習後 ×）: ['RAG（検索拡張生成）', '埋め込み（Embedding）', 'ベクトルデータベース']
A 学習後に講座の話にすり替えた: ['RLHF', 'DPO', 'プロンプトインジェクション']
B 学習後、2 通りとも × の用語: []
B 学習後、聞き方の片方だけ ○ の用語: []
B 退行（学習前 ○ → 学習後 ×）: []
C 学習前は言えた → 学習後は分かったふり: ['lecNN-end タグ']
⑥ 学習時の聞き方でも ×（覚えていない）: []
⑥ 学習時の聞き方では ○、違う聞き方は 2 通りとも ×（応用が利かない）: []
⑥ 学習時の聞き方では ○、違う聞き方の片方だけ ○: []
平均の長さ「って何？」| 学習前 278 字 → 学習後 116 字
平均の長さ「の意味を一言で教えてください。」| 学習前 84 字 → 学習後 116 字


,質問,種類,学習前の出力,学習前 内容/わからない/講座化,学習後の出力,学習後 内容/わからない/講座化
0,トークンって何？,一般,トークンとは、通信や情報処理において、データを送信する際に使用される単位や要素のことです。\n\n例えば、メール送信では「メール」という言葉が、通信の単位として使われます。また、インターネット通信では「IP」というアドレスが、通信の単位として使われます。\n\nトークンは、通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使われることが多く、その定義は通信の単位として使,×/×/-,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,○/×/-
1,トークンの意味を一言で教えてください。,一般,トークンは、通信や情報交換において、メッセージを伝達するための単位です。,×/×/-,LLMがテキストを扱う最小単位。質問〜回答までを1つのトークンとして数える。間接質問（「何か説明してください？」）は1つの質問で複数回トークン数が増える。直接質問（「APIを呼ぶ」）は1回の質問でトークン数減る。,○/×/-
2,コンテキストウィンドウって何？,一般,コンテキストウィンドウは、テキストエディタやコードエディタなどのソフトウェアにおいて、ユーザーが特定の文脈（例：特定のファイル、特定のコード、特定の環境設定など）を参照して編集する機能です。\n\nこの機能は、ユーザーが「この文脈で何を操作するか」を明確にすることで、作業の効率を向上させ、エラーを減らすために設計されています。,×/×/-,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。,○/×/-
3,コンテキストウィンドウの意味を一言で教えてください。,一般,コンテキストウィンドウは、ユーザーが特定の文脈や状況に依存して情報を抽出・分析する機能です。,×/×/-,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。,○/×/-
4,ハルシネーションって何？,一般,ハルシネーションとは、光の波長が波長を短くする現象です。\n\nこの現象は、光の波長が波長を短くする現象を指します。光の波長が波長を短くする現象は、ハルシネーションと呼ばれます。\n\n光の波長が波長を短くする現象は、ハルシネーションと呼ばれます。,×/×/-,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,○/×/-
...,...,...,...,...,...,...
72,「MCPサーバー」とは何ですか？,講座固有・学習時の聞き方,「MCPサーバー」は、現時点では実在する用語ではありません。\n\nMCP（Model Context Protocol）は、大規模言語モデル（LLM）や AI 技術において、モデルとユーザーとの通信を標準化するためのプロトコルとして提案されたものです。しかし、そのプロトコル自体が「サーバー」という概念を持つわけではありません。\n\nMCP 技術は主に以下の 2 つのプラットフォームで実装されています：\n1. **MCP Server**：これは、MCP プロトコルを定義する**サーバー**（サーバー）です。\n2. **MCP Client**：これは、MCP プロトコルを定義するクライアント（クライアント）です。\n\nしたがって、「MCP サーバー」という用語は、MCP プロトコルを定義するサーバーという概念自体を指すのではなく、MCP 技術全体を指す言葉として使われる,×/×/-,MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。,○/×/-
73,「mcpServers設定」とは何ですか？,講座固有・学習時の聞き方,「mcpServers」は、MCP（Model Context Protocol）のサーバー設定を指します。\n\nこれは、モデルコンテキストプロトコル（MCP）のサーバーを定義するパラメータです。具体的には、モデルコンテキストプロトコルを定義するサーバーのリストを指定するパラメータです。\n\nこの設定は、モデルコンテキストプロトコルを定義するサーバーのリストを指定するパラメータです。,×/×/-,使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。,○/×/-
74,「ツールの名前空間」とは何ですか？,講座固有・学習時の聞き方,「ツールの名前空間」とは、ソフトウェア開発において、特定のツール（例：Python の `requests`、Java の `Spring Boot` など）の機能や特性を定義する際に使用される概念です。\n\nこの空間には、ツールが持つ特徴や機能、用途などを記述する定義文（定義文）が含まれます。例えば、`requests` ツールは「HTTP 通信を処理する」などの定義文がその名前空間に含まれます。\n\nただし、この用語は実在しているわけではありません。ツールは実在するが、その「名前空間」自体は定義文として存在しないため、この用語はわかりません。,×/○/-,複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。,○/×/-
75,「fetchサーバー」とは何ですか？,講座固有・学習時の聞き方,「fetchサーバー」は、HTTP 通信において、クライアント（ユーザーやアプリケーション）がサーバーからデータを取得する際に使用される、主に HTTP 1.1 標準の「Fetch」プロトコルを定義するサーバーです。\n\nこのプロトコルは、クライアントがサーバーからデータを取得する際に、サーバーが直接そのデータを返すのではなく、**クライアントがサーバーからデータを取得する際、そのデータを取得する際に使用される**という特徴を持っています。つまり、サーバーが直接データを返すのではなく、クライアントがサーバーからデータを取得する際に使用されるという定義です。\n\nこのプロトコルは、HTTP 1.1 標準の Fetch プロトコルを定義するサーバーです。,×/×/-,指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。,○/×/-


Drive に保存しました: /content/drive/MyDrive/aie2026/qlora_lab/glossary


In [8]:
# 9-7. 採点の見直し（再生成はしない。保存済みの結果を、見直した採点で採点し直す）
# 結果を見てから採点を変えたので、元の採点（9-4・9-6 の要約）と並べて報告する。変えたところと理由:
#  ① B: must_include は定義の末尾の固有語だけを見ていた → 「1 文目が定義の 1 文目とどれだけ似ているか」を別に数える
#  ② わからない: 定義を作ったあとに system の文をなぞって「わかりません」と書く出力まで ○ にしていた → 1 文目で断ったときだけ ○
#  ③ 講座化: 「第8回」などの回番号を拾えず、質問文の語（report.md の AIE2026）をなぞるだけで ○ になっていた → 回番号を正規表現で拾い、質問文にある語は除く
#  ⑤ A: 学習前の長い回答は、後半でたまたまキーワードに当たりやすかった → 1 文目だけで判定する（最終的な判断は下の表を目で見て付ける）
from difflib import SequenceMatcher
HEAD_TH = 0.6  # 1 文目の一致度の閾値（模範と同じなら 1.0）

def first_sentence(s): return re.split(r"(?<=。)", s.strip())[0]
def head_sim(raw, term): return SequenceMatcher(None, first_sentence(raw), first_sentence(GLOSSARY[term])).ratio()
def refused(raw): return any(w in first_sentence(raw) for w in UNSURE_MARKERS)
def course_link2(raw, q):
    hits = [w for w in ["本講座", "AIE2026", "講座repo"] if w in raw and w not in q]
    hits += [m for m in re.findall(r"第\s*\d+\s*回", raw) if m not in q]
    return bool(hits)
def a_first(r): return all(any(w in first_sentence(r["raw"]) for w in g) for g in r["must_include"])

def rescore(rows, label):
    def cnt(kinds, f):
        rs = [r for r in rows if r["kind"] in kinds]
        return f"{sum(bool(f(r)) for r in rs)}/{len(rs)}"
    return {"label": label,
            "B 講座固有: 1文目が定義と一致": cnt(("講座固有",), lambda r: head_sim(r["raw"], r["term"]) >= HEAD_TH),
            "B 講座固有: 末尾の固有語（元の採点）": cnt(("講座固有",), lambda r: r["content_ok"]),
            "B 学習時の聞き方: 1文目が定義と一致": cnt(("講座固有・学習時の聞き方",), lambda r: head_sim(r["raw"], r["term"]) >= HEAD_TH),
            "B 学習時の聞き方: 末尾の固有語": cnt(("講座固有・学習時の聞き方",), lambda r: r["content_ok"]),
            "B 一般: 1文目が定義と一致": cnt(("一般",), lambda r: head_sim(r["raw"], r["term"]) >= HEAD_TH),
            "A 1文目にキーワード": cnt(("実在・用語集外",), a_first),
            "A 講座化（見直し）": cnt(("実在・用語集外",), lambda r: course_link2(r["raw"], r["question"])),
            "C 1文目で断った": cnt(("架空",), lambda r: refused(r["raw"])),
            "C 講座化（見直し）": cnt(("架空",), lambda r: course_link2(r["raw"], r["question"])),
            "A 1文目で断った（断りすぎ）": cnt(("実在・用語集外",), lambda r: refused(r["raw"])),
            "B 1文目で断った（断りすぎ）": cnt(("一般", "講座固有", "講座固有・学習時の聞き方"), lambda r: refused(r["raw"]))}

# system なし（前回）と system あり（今回）の学習前後を、同じ見直し後の採点で並べる
runs = {}
for name in ["glossary_before", "glossary_after", f"glossary_{G_TAG}_before", f"glossary_{G_TAG}_after", f"glossary_{G_RUN}_after"]: # 5 エポック（sys_after）と 20 エポック（sys_e20_after）を並べる
    p = next((q for q in [GW / f"eval_{name}.json", GDRIVE / f"eval_{name}.json"] if q.exists()), None)
    if p: runs[name] = json.loads(p.read_text(encoding="utf-8"))["rows"]
rescored = pd.DataFrame([rescore(rows, name) for name, rows in runs.items()]).set_index("label").T
display(rescored)
(GW / "rescore_glossary.json").write_text(rescored.to_json(force_ascii=False, indent=2), encoding="utf-8")
if USE_DRIVE: shutil.copy(GW / "rescore_glossary.json", GDRIVE / "rescore_glossary.json")

# ⑤ A の目視用: system ありの学習前後の回答を並べる（○×は自分で付ける）
b_rows, a_rows = runs.get(f"glossary_{G_TAG}_before", []), runs.get(f"glossary_{G_RUN}_after", [])
display(pd.DataFrame([{"用語": b["term"], "学習前": b["raw"], "学習後": a["raw"]}
                      for b, a in zip(b_rows, a_rows) if b["kind"] == "実在・用語集外"]))

label,glossary_before,glossary_after,glossary_sys_before,glossary_sys_after,glossary_sys_e20_after
B 講座固有: 1文目が定義と一致,0/30,19/30,0/30,14/30,30/30
B 講座固有: 末尾の固有語（元の採点）,0/30,2/30,0/30,2/30,30/30
B 学習時の聞き方: 1文目が定義と一致,0/0,0/0,0/15,7/15,15/15
B 学習時の聞き方: 末尾の固有語,0/0,0/0,0/15,1/15,15/15
B 一般: 1文目が定義と一致,0/14,9/14,0/14,12/14,14/14
A 1文目にキーワード,4/10,2/10,4/10,1/10,1/10
A 講座化（見直し）,0/10,4/10,0/10,2/10,3/10
C 1文目で断った,0/8,0/8,0/8,0/8,0/8
C 講座化（見直し）,1/8,3/8,0/8,3/8,4/8
A 1文目で断った（断りすぎ）,0/10,0/10,0/10,0/10,0/10


,用語,学習前,学習後
0,RAG（検索拡張生成）,RAG（検索拡張生成）とは、検索エンジンや知識ベースから取得した情報を「検索」として利用し、その結果を生成する技術です。\n\nこの技術は、従来の生成モデル（LLM）が「知識がない」という理由で回答が無視される問題を解決するために開発されました。具体例として、特定の分野（例：医学、法律、技術）の専門知識に基づいて回答を生成するようになります。\n\nただし、この技術は「知識がない」という前提でしか機能しないため、知識がない場合や、知識が不明確な場合、回答は「その用語はわかりません。」と答える必要があります。,LLMの出力を信頼できる事実や文章に丸ごと取り込む仕組み。検索をツールで補完する（ベースのRAG）や、モデルをデータで補完する（アダプタのRAG）。
1,埋め込み（Embedding）,埋め込み（Embedding）は、テキストや画像などのデータに意味を付与し、その意味を表現するベクトル（数値）を生成する技術です。,モデルがテキストを座標で表し、別のテキストに同じモデルで再度認識したときの結果を同じモデルで再度テキストに直す処理。LLMの性能を上げるために多くのトークンに分割したテキストで作業し、結果は1文字だけで受け取る設計。
2,Transformer,Transformer は、機械学習において非常に重要な概念です。\n\n**Transformer** は、**「自発的（自己）な」または「自己決定」な** ことを意味します。\n\nこの言葉は、**「自己決定性（Autonomy）」** という概念を指します。つまり、システムやプロセスが「自分自身で決定する」のではなく、外部の指示や外部のルールに従って行動するのではなく、**「自分自身で判断し、行動する」**という性質を指します。\n\n例えば、AI の学習プロセスや、システムがどのように動くかという点では、外部の指令に従って行動するのではなく、**「自分自身で判断し、行動する」**という特性を表現しています。\n\nしたがって、質問された用語の意味は以下の通りです。\n\n**「自己決定（Autonomy）」**,LLMの周りを、モデルが直接体験できる仕組みとして考えられる。6.5.0のPiはベースモデルに単純にテキストを書ительно、ツールを呼び出してデータで回す、画像をモデルに渡してモデルが見る、など多くのモードで使える。
3,アテンション（注意機構）,アテンションは、人間の注意を集中させるための機能です。,モデルの眼の位置を定めるためのハイパーパラメータ。焦点を絞るための指示文で、LLMが文章を読み、回答を返すための指示文として使う。
4,ベクトルデータベース,ベクトルデータベースは、データが「ベクトル（矢）」の形をとった形式で保存・検索・処理されるデータベースです。\n\nベクトルは、数値の大きさと方向を同時に表現するもので、例えば「100 点の x 座標と y 座標」を一つのベクトルとして表現できます。ベクトルデータベースでは、このベクトル形式のデータが主キーとして管理され、検索や操作を行う際、そのベクトルを特定して処理します。\n\nしたがって、ベクトルデータベースは、**「ベクトル（矢）の形をとったデータ形式で、そのベクトルを特定して検索・処理するデータベース」**と定義されます。,大量のテキストや画像を、それぞれの項目に付ける векクトで、間接質問やリクエスト・返信で検索するシステム。GoogleのBERTでモデルが付けられた。mcpServers設定でツールが有名になる。
5,RLHF,RLHF は Reinforcement Learning from Human Feedback の略です。,事前学習済みのモデルを、人間との対話で良い回答を学ぶ手法。LoRAで学習し、結果（スコア）を評価し、別のモデルやツールでスコアを上げる。本講座の第2回演習ではモデルをHuman-in-the-loopで評価し、Scoreを上げる。
6,DPO,DPO（Deep Packet Inspection）とは、ネットワークの通信データを「パケット」のレベルで分析し、その内容（通信内容）を特定・解析する技術です。\n\nこの技術は、ネットワークのセキュリティや信頼性を高めるために、特定の通信内容（例：ウイルス、スパムメール、不正アクセスなど）を検出・ブロックする際に使われます。,LLMがツールとの関係や行動の方向を、別のモデルやデータで定義し、その定義をツールや行動の戻り値に付ける、然后再使用中のモデルで実現する学習方法。第3回の回数を減らせる。
7,知識蒸留,知識蒸留は、知識を蒸発させること。,学習済みの重みを、別のモデルやデータで培った新しい軽量モデルに置き換えること。モデルが同じにすることから「同じツール」と数値が同じにすることから「同じデータ」と考える。
8,プロンプトインジェクション,プロンプトインジェクションとは、特定のユーザーの意図や文脈を、AI の生成プロセスに組み込むことで、より自然で有用な回答を生成する技術です。\n\nこの技術は、AI が「ユーザーの言葉」をそのまま受け止め、その意図を尊重しながら、より深く理解した回答を出力する仕組みを指します。例えば、ユーザーが「この文章を日本語で書き直してください」という指示をプロンプトインジェクションとして入力すると、AI はその意図を尊重し、元の文章のニュアンスや文脈を考慮して、より自然な日本語で再構成する回答を生成します。\n\nただし、この技術はあくまで「ユーザーの意図を尊重する」ものであり、AI が「ユーザーの言葉」をそのままそのまま出力するわけではありません。AI は、ユーザーの意図を尊重しつつ、その意図を尊重した上で、より,LLMがツールを呼び出すための入力。指定したツールの名前・説明文・引数のスキーマで、ツールが認識され、呼び出しが受け入れる。本講座では「システムプロンプト」がプロンプトインジェクションになる。
9,Few-shot プロンプト,Few-shot プロンプトの意味は、学習されたモデルが「提示された 1 つの例文（Few-shot）」と「1 つの例文（Few-shot）」の組み合わせから、新しい文脈や質問に対して、その組み合わせの文脈を学習して回答する能力を指します。,1つの例と2つの次の例（または図の2つ）に、モデルが次のトークンを指示する入力。LLMが学習していないため、指示の順序や形式に依存する。


### 9-8. この検証方法の限界

**要約：この検証で言えるのは「Qwen3.5-0.8B に、この 1 つの設定（r=16・5 エポック・seed 1 つ）で用語集を学習させたとき、この 77 問で定義文をどれだけ再現できたか」まで。LoRA 一般や「知識を理解したか」について結論を出すには、問数・採点・比較条件の 3 点が足りない。**

**1. 問数と再現性**
- A（実在・用語集外）は 10 問、C（架空）は 8 問しかない。1〜2 問の増減は誤差と区別できない。
- 学習は 1 回（seed 1 つ）、生成は貪欲生成 1 回だけ。GPU での学習は完全には再現しないので、同じ設定でも数字が少し変わりうる。

**2. 採点の妥当性**
- 内容の判定はキーワードが含まれるかどうか。説明が正しいかではなく、語句が出たかを見ている。B の講座固有語は定義の末尾の固有語なので、定義を丸暗記すれば ○、要点を言い換えて正しく説明すると × になりうる。
- 9-7 の採点の見直し（1 文目だけで判定、一致度の閾値 0.6 など）は結果を見てから決めた。見直し後の数字は、見つけた失敗を拾いやすい方向に寄っている可能性がある。
- 「わからない」の判定は「わかりません」の文字列だけを見ている。system で断り方を固定したので、測っているのは「知らないと見分けられるか」と「system の指示に従うか」が混ざったもの。
- 「一般」と「講座固有」の振り分け（Codespaces secret など）は自分の判断で、境界があいまい。

**3. 観点ごとの限界**
- A（忘却）：調べたのは「用語の意味を答える」タスクだけ。会話・推論・本編の JSON 形式など、他の能力が落ちたかは見ていない。また学習前の正答が 4/10 と少なく、忘却を測る余地自体が小さい。学習前の答えは長いので、後半でたまたまキーワードに当たって ○ になりやすい。
- B（応用）：違う聞き方は 2 通りのテンプレートだけで、学習データの聞き方に近い。言い換えへの強さは見られるが、覚えた知識を使って別の問い（例：その仕組みを使うとき何に気をつけるか）に答えられるかは見ていない。
- C（分かったふり）：架空語は自分で作ったもので、lecNN-start に対する lecNN-end のように本物に似せた語が多い。似ていない架空語や、講座と無関係の架空語での振る舞いは分からない。

**4. 比較条件**
- 学習時の聞き方でも講座固有語をほとんど答えられていない（学習後 1/15）。つまり今回の設定ではそもそも覚えきれていない可能性が高く、「LoRA では知識を覚えられない」のか「学習量が足りない」のかを切り分けられない。
- 学習の設定（エポック数・r・学習率）は 1 点だけ。忘却や分かったふりが学習量でどう変わるかは見ていない。
- system なし（前回）と system あり（今回）の比較では、system が学習データと評価の両方に入っている。差が「指示を足した効果」か「指示つきで学習した効果」かは切り分けにくい。
- モデルは 0.8B の 1 つだけ。大きいモデルや、もともと知識の多いモデルで同じ結果になるかは分からない。